# 01 — Chargement et exploration du dataset OPUS-100 EN→FR

Dataset : [Helsinki-NLP/opus-100](https://huggingface.co/datasets/Helsinki-NLP/opus-100)  
Subset : `en-fr` — 1M paires train, format Parquet  
Tâche : traduction anglais → français

In [ ]:
from datasets import load_dataset

ds = load_dataset("Helsinki-NLP/opus-100", "en-fr")
ds

DatasetDict({
    test: Dataset({
        features: ['translation'],
        num_rows: 2000
    })
    train: Dataset({
        features: ['translation'],
        num_rows: 1000000
    })
    validation: Dataset({
        features: ['translation'],
        num_rows: 2000
    })
})

## Structure d'un exemple

In [3]:
print("Splits disponibles :", list(ds.keys()))
print()

# TODO : afficher le nombre de paires pour chaque split
for split, data in ds.items():
    print(f'taille de {split} est {len(data)}')

Splits disponibles : ['test', 'train', 'validation']

taille de test est 2000
taille de train est 1000000
taille de validation est 2000


In [6]:
train = ds["train"]

# TODO : afficher les 5 premières paires EN → FR
#        chaque exemple est dans train[i]["translation"] → dict {"en": ..., "fr": ...}
for i in range(5):
    pair = train[i]["translation"]
    print(pair)

{'en': 'The time now is 05:08 .', 'fr': 'The time now is 05:05 .'}
{'en': 'This Regulation shall enter into force on the seventh day following its publication in the Official Journal of the European Union.', 'fr': "Le présent règlement entre en vigueur le septième jour suivant celui de sa publication au Journal officiel de l'Union européenne."}
{'en': "Hello, what's that?", 'fr': "Qu'est-ce que c'est que ça ?"}
{'en': 'And then I will teach you everything i know.', 'fr': "Et alors, je t'apprendrai tout ce que je sais."}
{'en': 'Did you find something?', 'fr': 'Par ici !'}


## Extraction des phrases source et cible

On travaille sur un sous-ensemble pour l'exploration (le train complet = 1M).

In [ ]:
N = 50_000

# TODO : extraire les N premières phrases source (EN) et cible (FR)
#        hint : [ex["translation"]["en"] for ex in train.select(range(N))]
src_sentences = [ex["translation"]["en"] for ex in train.select(range(N))]
trg_sentences = [ex["translation"]["fr"] for ex in train.select(range(N))]

print(f"Exemples chargés : {len(src_sentences):,}")

Exemples chargés : 50,000


## Tokenisation avec spaCy

spaCy sépare la ponctuation des mots → vocab plus propre que `.split()`.  
Ex : `"Hello, world!"` → `["hello", ",", "world", "!"]`

In [ ]:
import spacy
from tqdm import tqdm

# TODO : charger les modèles spaCy pour l'anglais et le français
#        désactiver "ner" et "parser" (inutiles ici, gain de vitesse)
nlp_en = spacy.load("en_core_web_sm", disable=["ner", "parser"])
nlp_fr = spacy.load("fr_core_news_sm", disable=["ner", "parser"]) 

def batch_tokenize(nlp, sentences, batch_size=512):
    """Tokenise en batch via nlp.pipe()."""
    # TODO : itérer sur nlp.pipe(sentences, batch_size=batch_size)

    return [
    [tok.text.lower() for tok in doc]   # liste de tokens pour UN doc
    for doc in tqdm(nlp.pipe(sentences, batch_size = batch_size), total= len(sentences))]  # pour chaque doc
    #        pour chaque doc, garder les tokens non-espaces en lowercase
    #        utiliser tqdm pour la barre de progression

src_tokens = batch_tokenize(nlp_en, src_sentences)
trg_tokens = batch_tokenize(nlp_fr, trg_sentences)

100%|██████████| 50000/50000 [01:18<00:00, 636.38it/s]

Exemple tokenisé :
EN : ['the', 'time', 'now', 'is', '05:08', '.']
FR : ['the', 'time', 'now', 'is', '05:05', '.']


In [ ]:
print("Exemple tokenisé :")
print(f"EN : {src_tokens[:5]}")
print(f"FR : {trg_tokens[:5]}")

Exemple tokenisé :
EN : [['the', 'time', 'now', 'is', '05:08', '.'], ['this', 'regulation', 'shall', 'enter', 'into', 'force', 'on', 'the', 'seventh', 'day', 'following', 'its', 'publication', 'in', 'the', 'official', 'journal', 'of', 'the', 'european', 'union', '.'], ['hello', ',', 'what', "'s", 'that', '?'], ['and', 'then', 'i', 'will', 'teach', 'you', 'everything', 'i', 'know', '.'], ['did', 'you', 'find', 'something', '?']]
FR : [['the', 'time', 'now', 'is', '05:05', '.'], ['le', 'présent', 'règlement', 'entre', 'en', 'vigueur', 'le', 'septième', 'jour', 'suivant', 'celui', 'de', 'sa', 'publication', 'au', 'journal', 'officiel', 'de', "l'", 'union', 'européenne', '.'], ["qu'", 'est', '-ce', 'que', "c'", 'est', 'que', 'ça', '?'], ['et', 'alors', ',', 'je', "t'", 'apprendrai', 'tout', 'ce', 'que', 'je', 'sais', '.'], ['par', 'ici', '!']]


## Distribution des longueurs

In [16]:
import math
src_lens = [len(s) for s in src_tokens]
trg_lens = [len(s) for s in trg_tokens]

# TODO : afficher longueur moyenne et max pour EN et FR
f'longeur moyenne des textes anglais est {sum(src_lens)/len(src_lens)} et pour le francais est {sum((trg_lens))/len(trg_lens)} '

'longeur moyenne des textes anglais est 17.4585 et pour le francais est 19.15962 '

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# TODO : tracer l'histogramme des longueurs EN sur axes[0]  (couleur : steelblue)

# TODO : tracer l'histogramme des longueurs FR sur axes[1]  (couleur : tomato)
# TODO : ajouter titres et labels

plt.tight_layout()
plt.show()

## Taille du vocabulaire

In [17]:
import sys
sys.path.append("../src")

from data import Vocab

# TODO : construire le vocab source et cible avec min_freq=2
src_vocab = Vocab.build(src_tokens, min_freq = 2 )
trg_vocab = Vocab.build(trg_tokens, min_freq=2)

print(f"Vocab EN (min_freq=2) : {len(src_vocab):,} tokens")
print(f"Vocab FR (min_freq=2) : {len(trg_vocab):,} tokens")

Vocab EN (min_freq=2) : 19,962 tokens
Vocab FR (min_freq=2) : 23,920 tokens


In [ ]:
from collections import Counter

src_freq = Counter(t for s in src_tokens for t in s)
trg_freq = Counter(t for s in trg_tokens for t in s)

# TODO : afficher le top 10 des tokens les plus fréquents pour EN et FR
